<a href="https://colab.research.google.com/github/123321juliocesar/fundamentos_IA/blob/main/ejercicio_red_bayesiana_aprobacion_estudiante.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio de red bayesiana

nombre: Julio Cesar Gonzales Castillo

Simulación para determinar la probabilidad de que un estudiante apruebe una asignatura considerando sus horas de estudio, asistencia y nota de examen.




In [2]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.3 MB/s eta 0:00:00


In [3]:
# libreria de redes bayesianas
# crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork
# crear una tabla de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD
# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [4]:
# crear la red bayesiana
model = DiscreteBayesianNetwork([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion")
])


In [5]:
# crear la tabla de probabilidades condicionales para HorasEstudio
# 40% de probabilidad de estudiar muchas horas y 60% de estudiar pocas horas
cpd_horas_estudio = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.4], [0.6]],
    state_names={"HorasEstudio": ["Muchas", "Pocas"]}
)


In [6]:
# crear la tabla de probabilidades condicionales para Asistencia
# si estudia muchas horas: 80% de asistencia alta y 20% baja
# si estudia pocas horas: 40% de asistencia alta y 60% baja
cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.8, 0.4], [0.2, 0.6]],
    evidence=["HorasEstudio"],
    evidence_card=[2],
    state_names={
        "Asistencia": ["Alta", "Baja"],
        "HorasEstudio": ["Muchas", "Pocas"]
    }
)


In [7]:
# crear la tabla de probabilidades condicionales para NotaExamen
# con asistencia alta: 75% de nota alta y 25% baja
# con asistencia baja: 30% de nota alta y 70% baja
cpd_nota_examen = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[[0.75, 0.30], [0.25, 0.70]],
    evidence=["Asistencia"],
    evidence_card=[2],
    state_names={
        "NotaExamen": ["Alta", "Baja"],
        "Asistencia": ["Alta", "Baja"]
    }
)


In [8]:
# crear la tabla de probabilidades condicionales para Aprobacion
# con nota alta: 90% aprueba y 10% desaprueba
# con nota baja: 20% aprueba y 80% desaprueba
cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[[0.90, 0.20], [0.10, 0.80]],
    evidence=["NotaExamen"],
    evidence_card=[2],
    state_names={
        "Aprobacion": ["Aprueba", "Desaprueba"],
        "NotaExamen": ["Alta", "Baja"]
    }
)


In [9]:
# agregar al modelo las tablas de probabilidades condicionales
model.add_cpds(
    cpd_horas_estudio,
    cpd_asistencia,
    cpd_nota_examen,
    cpd_aprobacion
)
model.check_model()


True

In [10]:
print("Modelo de red bayesiana creado y verificado correctamente.")
print(model.edges())


Modelo de red bayesiana creado y verificado correctamente.
[('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


In [11]:
# Inferencia en la red bayesiana
# motor o proceso de inferencia para el grafo
inferencia = VariableElimination(model)

# Probabilidad de aprobar si el estudiante estudia muchas horas
resultado_aprobacion = inferencia.query(
    variables=["Aprobacion"],
    evidence={"HorasEstudio": "Muchas"}
)

print("Probabilidad de aprobar dado que el estudiante estudia muchas horas:")
print(resultado_aprobacion)


Probabilidad de aprobar dado que el estudiante estudia muchas horas:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Aprueba)    |            0.6620 |
+------------------------+-------------------+
| Aprobacion(Desaprueba) |            0.3380 |
+------------------------+-------------------+


In [12]:
# Probabilidad de aprobar sin evidencia
resultado_aprobacion_2 = inferencia.query(
    variables=["Aprobacion"]
)

print("Probabilidad de aprobar sin evidencia:")
print(resultado_aprobacion_2)


Probabilidad de aprobar sin evidencia:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Aprueba)    |            0.5864 |
+------------------------+-------------------+
| Aprobacion(Desaprueba) |            0.4136 |
+------------------------+-------------------+


In [13]:
# Probabilidad de aprobar si tiene asistencia alta
resultado_aprobacion_3 = inferencia.query(
    variables=["Aprobacion"],
    evidence={"Asistencia": "Alta"}
)

print("Probabilidad de aprobar dado que el estudiante tiene asistencia alta:")
print(resultado_aprobacion_3)


Probabilidad de aprobar dado que el estudiante tiene asistencia alta:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Aprueba)    |            0.7250 |
+------------------------+-------------------+
| Aprobacion(Desaprueba) |            0.2750 |
+------------------------+-------------------+


In [14]:
# Probabilidad de aprobar si obtiene una nota alta
resultado_aprobacion_4 = inferencia.query(
    variables=["Aprobacion"],
    evidence={"NotaExamen": "Alta"}
)

print("Probabilidad de aprobar dado que el estudiante obtiene una nota alta:")
print(resultado_aprobacion_4)


Probabilidad de aprobar dado que el estudiante obtiene una nota alta:
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Aprueba)    |            0.9000 |
+------------------------+-------------------+
| Aprobacion(Desaprueba) |            0.1000 |
+------------------------+-------------------+
